# From a Langmuir isotherm to a macrostate particle distribution

**BEA, 552 K, binary n-heptane / 2-methylhexane.**

Almost every mixture breakthrough calculation starts the same way: fit an
analytical isotherm to each pure component, then bolt on a mixing rule — IAST,
or a competitive extension of the isotherm — to get the multicomponent
equilibrium. The mixing rule is where the modelling assumptions hide.

A **macrostate particle distribution (MPD)** removes that step. Instead of an
isotherm, the equilibrium model is a probability distribution over
particle-number vectors

$$\mathbf n = \begin{bmatrix} n_1 & n_2 & \cdots & n_{N_c}\end{bmatrix}^{\mathsf T},
\qquad N = \sum_j n_j,$$

recorded once at a reference state $(f^{\mathrm{ref}}, T^{\mathrm{ref}}, V)$ —
in practice from NVT+$W$ or transition-matrix Monte Carlo. Ruptura *reweights*
that one distribution to whatever fugacity, composition and temperature the
column solver asks for, and reads the loading straight off the mean:

$$q_j^{*}(f,\mathbf y,T)=\frac{\langle N_j\rangle_{f,\mathbf y,V,T}}{N_{\mathrm A}m_{\mathrm{fw}}}.$$

There is no mixing rule, and no fitted mixture isotherm.

This notebook makes that concrete without needing a Monte Carlo distribution.
A dual-site Langmuir isotherm **is already** a statement about a distribution
over site occupancies, so we can write the corresponding MPD down in closed
form, hand it to Ruptura, and check that the column solver reproduces the
analytical result. The story runs:

1. fit the BEA GCMC isotherms in the form a multinomial requires;
2. turn that fit into a two-component, two-site-family multinomial, and look
   at it;
3. watch reweighting move the distribution as the pressure rises;
4. compare MPD isotherms against dual-site Langmuir + IAST;
5. compare MPD and Langmuir/IAST breakthrough curves in the same column;
6. count the cost.

Everything in the example is generated from three scripts, so nothing can drift
apart: `fit_isotherms.py` → `fitted_parameters.json`,
`generate_particle_distribution.py` → `particle_distribution.data`, and
`generate_simulations.py` → the four `simulation.json` inputs.

In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

EXAMPLE_DIR = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "generate_particle_distribution.py").is_file()
    ),
    None,
)
if EXAMPLE_DIR is None:
    raise FileNotFoundError("Run this notebook from examples/BEA-alkanes-C7/mpd or one of its subdirectories.")

REPO_ROOT = next(
    parent for parent in EXAMPLE_DIR.parents if (parent / "CMakeLists.txt").is_file() and (parent / "src").is_dir()
)
FITTING_DIR = EXAMPLE_DIR.parent / "fitting"
BINARY = Path(os.environ.get("RUPTURA_BIN", REPO_ROOT / "build" / "ruptura")).expanduser()

GAS_CONSTANT = 8.31446261815324  # J mol^-1 K^-1

TOTAL_TIME_PATTERN = re.compile(r"^Total time:\s+([0-9.eE+-]+) s$", flags=re.MULTILINE)
SUB_TIMING_PATTERN = re.compile(r"^\s{2}([A-Za-z]+):\s+([0-9.eE+-]+) s$", flags=re.MULTILINE)


def run_script(name: str) -> str:
    completed = subprocess.run(
        [sys.executable, str(EXAMPLE_DIR / name)], check=True, capture_output=True, text=True, cwd=EXAMPLE_DIR
    )
    return completed.stdout.strip()


def run_ruptura(directory: Path, *, force: bool = False) -> dict:
    '''Run Ruptura in `directory`, caching the result in benchmark.json.'''
    directory = Path(directory)
    record_path = directory / "benchmark.json"
    if record_path.is_file() and not force:
        return json.loads(record_path.read_text(encoding="utf-8"))
    if not BINARY.is_file():
        raise FileNotFoundError(
            f"Ruptura executable not found at {BINARY}. Build it, or set the RUPTURA_BIN environment variable."
        )

    log_path = directory / "ruptura.log"
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as stream:
        completed = subprocess.run([str(BINARY)], cwd=directory, stdout=stream, stderr=subprocess.STDOUT, text=True)
    wall_time = time.perf_counter() - started
    if completed.returncode != 0:
        raise RuntimeError(f"Ruptura failed in {directory} (exit {completed.returncode}); see {log_path}")

    log = log_path.read_text(encoding="utf-8", errors="replace")
    total = TOTAL_TIME_PATTERN.findall(log)
    record = {
        "directory": directory.name,
        "wall_time_s": wall_time,
        "total_time_s": float(total[-1]) if total else float("nan"),
        "sub_timings_s": {name: float(value) for name, value in SUB_TIMING_PATTERN.findall(log)},
    }
    record_path.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
    return record


def read_mixture_prediction(directory: Path, index: int, component: str) -> dict[str, np.ndarray]:
    table = np.loadtxt(Path(directory) / f"component_{index}_{component}.data", comments="#")
    return {"pressure": table[:, 0], "pure": table[:, 1], "mixture": table[:, 2]}


def read_outlet(directory: Path, index: int, component: str) -> dict[str, np.ndarray]:
    '''Outlet history: dimensionless time, minutes, and p_i/(p_t y_i,0).'''
    directory = Path(directory)
    column_length = float(json.loads((directory / "simulation.json").read_text(encoding="utf-8"))["ColumnLength"])
    table = np.atleast_2d(np.loadtxt(directory / f"component_{index}_{component}.data", comments="#"))
    outlet = table[np.isclose(table[:, 2], column_length)]
    if outlet.size == 0:
        raise ValueError(f"No complete outlet profile in {directory}")
    order = np.argsort(outlet[:, 0])
    return {"tau": outlet[order, 0], "minutes": outlet[order, 1], "normalized": outlet[order, 12]}


def first_crossing(curve: dict[str, np.ndarray], level: float = 0.5) -> float:
    '''Time [min] of the first upward crossing of `level` (roll-up safe).'''
    values, times = curve["normalized"], curve["minutes"]
    above = np.flatnonzero(values >= level)
    if above.size == 0:
        return float("nan")
    index = int(above[0])
    if index == 0:
        return float(times[0])
    span = values[index] - values[index - 1]
    if span <= 0.0:
        return float(times[index])
    return float(times[index - 1] + (level - values[index - 1]) / span * (times[index] - times[index - 1]))


print(f"Example directory: {EXAMPLE_DIR}")
print(f"Ruptura executable: {BINARY} ({'found' if BINARY.is_file() else 'MISSING'})")

## 1. Fitting the GCMC isotherms in the form a multinomial needs

`examples/BEA-alkanes-C7/fitting` holds GCMC results for the C7 isomers on BEA
at 552 K. We keep two of them: the linear **n-heptane (nC7)**, which BEA
adsorbs most strongly, and the singly branched **2-methylhexane (C6m2)**,
`Results.dat-BEA-Repeat-552K-2mC6`. Each file gives the fugacity (column 3),
the absolute loading in mol/kg (column 8) with its error bar, and the simulated
heat of desorption in kelvin (column 20).

The fit has one unusual constraint. A multinomial describes $N_s$
*indistinguishable sites competing for the same molecules*, so within a site
family the saturation capacity has to be **shared** by the components — they can
differ in affinity, not in how many sites exist. `fit_isotherms.py` therefore
fits both components at once,

$$q_i(f)=\sum_{s=1}^{2}\frac{q_{\mathrm{sat},s}\,b_{i,s}f}{1+b_{i,s}f},$$

with two shared capacities $q_{\mathrm{sat},s}$ and four affinities $b_{i,s}$,
by Nelder–Mead least squares on the absolute residuals. Imposing the constraint
during the fit rather than averaging two independent fits afterwards is the
difference between a model and a fudge.

$\Delta H$ is a different matter: **it cannot be fitted here at all.** A
single-temperature isotherm contains no information about the temperature
dependence. It is instead read from the GCMC heat of desorption, averaged over
the Henry-regime points,

$$Q^{\mathrm{eq}} = -R\,\bigl\langle \text{heat of desorption}\bigr\rangle,$$

which is the positive quantity Ruptura's `HeatOfAdsorption` expects in
$\kappa(T_s)=\kappa^{0}\exp\bigl(Q^{\mathrm{eq}}/RT_s\bigr)$.

In [ ]:
print(run_script("fit_isotherms.py"))

In [ ]:
sys.path.insert(0, str(EXAMPLE_DIR))
import generate_particle_distribution as builder  # noqa: E402

FITTED = json.loads((EXAMPLE_DIR / "fitted_parameters.json").read_text(encoding="utf-8"))
COMPONENTS = builder.COMPONENTS
AVOGADRO = builder.AVOGADRO
REFERENCE_FUGACITY = builder.REFERENCE_FUGACITY
REFERENCE_TEMPERATURE = builder.REFERENCE_TEMPERATURE
FRAMEWORK_MASS = builder.REFERENCE_FRAMEWORK_MASS
PARTICLES_PER_MOL_PER_KG = AVOGADRO * FRAMEWORK_MASS
SATURATION_CAPACITIES = builder.SATURATION_CAPACITIES
HEAT_OF_ADSORPTION = builder.HEAT_OF_ADSORPTION

GCMC = {}
for component, file_name in (("nC7", "Results.dat-BEA-Repeat-552K-nC7"),
                             ("C6m2", "Results.dat-BEA-Repeat-552K-2mC6")):
    table = np.loadtxt(FITTING_DIR / file_name, comments="#")
    GCMC[component] = {"fugacity": table[:, 2], "loading": table[:, 7], "error": table[:, 8]}


def affinity(component: str, site: int, temperature: float = REFERENCE_TEMPERATURE) -> float:
    '''b_{i,s}(T) = b_{i,s}(T_ref) exp(Q_eq/R (1/T - 1/T_ref)), Ruptura's temperature model.'''
    base = builder.SITE_FAMILIES[site][1][COMPONENTS.index(component)]
    exponent = HEAT_OF_ADSORPTION[component] / GAS_CONSTANT * (1.0 / temperature - 1.0 / REFERENCE_TEMPERATURE)
    return base * np.exp(exponent)


def pure_langmuir(component: str, fugacity, temperature: float = REFERENCE_TEMPERATURE):
    loading = 0.0
    for site, capacity in enumerate(SATURATION_CAPACITIES):
        b = affinity(component, site, temperature)
        loading = loading + capacity * b * fugacity / (1.0 + b * fugacity)
    return loading


def competitive_langmuir(fugacities, temperature: float = REFERENCE_TEMPERATURE) -> np.ndarray:
    '''Segregated competitive dual-site Langmuir loading [mol/kg].'''
    loading = np.zeros(len(COMPONENTS))
    for site, capacity in enumerate(SATURATION_CAPACITIES):
        affinities = [affinity(component, site, temperature) for component in COMPONENTS]
        denominator = 1.0 + sum(b * f for b, f in zip(affinities, fugacities))
        for index, (b, f) in enumerate(zip(affinities, fugacities)):
            loading[index] += capacity * b * f / denominator
    return loading


print(f"Framework mass represented by the distribution: {FRAMEWORK_MASS:.6e} kg")
print(f"  = {PARTICLES_PER_MOL_PER_KG:.3f} particles per mol/kg\n")
print(f"{'site family':>12} {'sites':>6} {'q_sat [mol/kg]':>15}"
      + "".join(f"{'b_' + component + ' [1/Pa]':>18}" for component in COMPONENTS))
for site, ((sites, affinities), capacity) in enumerate(zip(builder.SITE_FAMILIES, SATURATION_CAPACITIES), start=1):
    print(f"{site:>12} {sites:>6} {capacity:>15.5f}" + "".join(f"{b:>18.5e}" for b in affinities))
print(f"\nTotal sites: {builder.N_MAX}  ->  saturation capacity {sum(SATURATION_CAPACITIES):.4f} mol/kg")
for component in COMPONENTS:
    print(f"  Q_eq({component}) = {HEAT_OF_ADSORPTION[component] / 1000.0:.3f} kJ/mol")

### The fit, and what it does with temperature

The 552 K curves are the fit; the 473, 373 and 293 K curves are the same
parameters carried by $b_{i,s}(T)=b_{i,s}(T^{\mathrm{ref}})
\exp\!\bigl[Q^{\mathrm{eq}}/R\,(1/T-1/T^{\mathrm{ref}})\bigr]$ — Ruptura's
`HeatOfAdsorption` model, with the saturation capacities held fixed. Cooling
shifts each isotherm bodily to lower fugacity without changing its height,
which is the signature of a purely enthalpic affinity term.

Two caveats worth stating plainly. The shared-capacity constraint costs some
accuracy against the unconstrained per-component fit in
`fitting/BEA_fitted.json` — it is the price of a competitive site model. And
the GCMC heat of desorption in these files is nearly the same for ethane as for
n-heptane, which is not what one expects of a total isosteric heat, so the
sub-552 K curves should be read as an illustration of the temperature model
rather than as a prediction.

In [ ]:
TEMPERATURES = (552.0, 473.0, 373.0, 293.0)
temperature_colors = dict(zip(TEMPERATURES, plt.get_cmap("viridis")(np.linspace(0.05, 0.8, len(TEMPERATURES)))))
fugacity_grid = np.logspace(-1, 7, 400)

fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(12.5, 5.0), sharey=True, constrained_layout=True)
for axis, component in zip(axes, COMPONENTS):
    for temperature in TEMPERATURES:
        axis.plot(
            fugacity_grid, pure_langmuir(component, fugacity_grid, temperature),
            color=temperature_colors[temperature], linewidth=1.8,
            label=f"{temperature:g} K" + (" (reference)" if temperature == REFERENCE_TEMPERATURE else ""),
        )
    measured = GCMC[component]
    axis.errorbar(
        measured["fugacity"], measured["loading"], yerr=measured["error"],
        linestyle="none", marker="o", markersize=6, markerfacecolor="none",
        color="black", capsize=3, linewidth=1.0, label="GCMC, 552 K",
    )
    axis.set_xscale("log")
    axis.set_xlabel("fugacity  [Pa]")
    axis.set_title(component)
    axis.grid(alpha=0.2)
axes[0].set_ylabel("loading  [mol/kg]")
axes[0].legend(loc="upper left", fontsize=9)
fig.suptitle("Fitted dual-site Langmuir isotherms with shared saturation capacities")
plt.show()

print("Fit residuals at 552 K:")
for component in COMPONENTS:
    measured = GCMC[component]
    deviation = np.abs(pure_langmuir(component, measured["fugacity"]) - measured["loading"])
    print(f"  {component:>5}: max {deviation.max():.4f} mol/kg, "
          f"rms {np.sqrt((deviation ** 2).mean()):.4f} mol/kg")

## 2. The Langmuir isotherm *is* a multinomial

Take one site family: $N_s$ equivalent sites, each either empty or holding one
molecule of one component. In the grand-canonical ensemble the sites are
independent, so the number of ways of placing $\mathbf n$ molecules on them,
weighted by the activities $a_i=b_{i,s}f_i$, is a multinomial:

$$\Pi_s(\mathbf n)=\frac{N_s!}{\left(N_s-\sum_i n_i\right)!\,\prod_i n_i!}
\;\frac{\prod_i a_i^{\,n_i}}{\bigl(1+\sum_j a_j\bigr)^{N_s}}.$$

Its mean occupancy is $\langle n_i\rangle = N_s a_i/(1+\sum_j a_j)$, which
divided by $N_{\mathrm A}m_{\mathrm{fw}}$ is exactly the competitive Langmuir
term $q_{\mathrm{sat},s}b_{i,s}f_i/(1+\sum_j b_{j,s}f_j)$. **The multinomial is
not an approximation of the isotherm — it is the distribution the isotherm is
the mean of.**

Two independent site families give a *product* distribution, which we convolve
over the total counts $n_i=n_{i,1}+n_{i,2}$, because a macrostate records only
how many molecules of each species are in the framework, not where they sit.

### Choosing the framework mass

The particle-number axis is set by $m_{\mathrm{fw}}$: it must make
$N_s=q_{\mathrm{sat},s}N_{\mathrm A}m_{\mathrm{fw}}$ an integer number of sites,
and keep the counts inside the tabulated window. We fix the window at
$[0,200]$ and let the fit place the split: the two families take
$\lceil 200\,q_{\mathrm{sat},s}/\sum_r q_{\mathrm{sat},r}\rfloor$ sites each, and

$$m_{\mathrm{fw}}=\frac{200}{N_{\mathrm A}\sum_s q_{\mathrm{sat},s}}$$

follows. Rounding the split to integers moves the represented capacities by
less than 0.1 % from the fitted ones. No macrostate can then exceed 200
particles of either species, and the reference state sits comfortably in the
interior.

The file is tabulated with $\Delta n = 2$ — 101 nodes per species, 5151
populated macrostates instead of the 20301 a unit mesh would need. Section 6
shows what that costs: nothing over the pressure range used here, and a fraction
of a percent in the extreme Henry and saturation limits, where the distribution
becomes narrower than the mesh.

In [ ]:
print(run_script("generate_particle_distribution.py"))
print(run_script("generate_simulations.py"))

MESH, PI_REFERENCE = builder.reference_distribution()

# Cross-check the file Ruptura will read against the in-memory distribution.
stored = np.loadtxt(EXAMPLE_DIR / "particle_distribution.data", comments="#").reshape(PI_REFERENCE.shape)
print(f"\nmax |file - in-memory| = {np.abs(stored - PI_REFERENCE).max():.3e}")


def reweight(mole_fractions, fugacity, temperature=REFERENCE_TEMPERATURE):
    '''Reweight the reference MPD to (f, y, T); returns the normalized distribution.

    log Pi~(n) = log Pi_ref(n) + sum_j n_j log y_j + N log(f/f_ref) + N log(beta/beta_ref)

    The conditional-energy term of the full expression is absent because this
    distribution carries no <H> column: the model is isothermal by construction.
    '''
    first = MESH[:, None].astype(float)
    second = MESH[None, :].astype(float)

    log_weight = np.full(PI_REFERENCE.shape, -np.inf)
    live = PI_REFERENCE > 0.0
    log_weight[live] = np.log(PI_REFERENCE[live])
    log_weight = log_weight + (first + second) * (
        np.log(fugacity / REFERENCE_FUGACITY) + np.log(REFERENCE_TEMPERATURE / temperature)
    )
    for counts, fraction in ((first, mole_fractions[0]), (second, mole_fractions[1])):
        if fraction > 0.0:
            log_weight = log_weight + counts * np.log(fraction)
        else:
            log_weight = np.where(counts > 0.0, -np.inf, log_weight)

    finite = np.isfinite(log_weight)
    weight = np.zeros_like(PI_REFERENCE)
    weight[finite] = np.exp(log_weight[finite] - log_weight[finite].max())
    return weight / weight.sum()


def mean_particle_numbers(distribution):
    return np.array([distribution.sum(axis=1) @ MESH, distribution.sum(axis=0) @ MESH], dtype=float)


def mpd_loading(mole_fractions, fugacity):
    return mean_particle_numbers(reweight(mole_fractions, fugacity)) / PARTICLES_PER_MOL_PER_KG


reference_mean = mean_particle_numbers(PI_REFERENCE)
print(f"\nReference state ({REFERENCE_FUGACITY:g} Pa each, {REFERENCE_TEMPERATURE:g} K):")
for component, count in zip(COMPONENTS, reference_mean):
    print(f"  <n_{component}> = {count:7.3f} particles = {count / PARTICLES_PER_MOL_PER_KG:.4f} mol/kg")

## 3. Looking at the distribution

Below is $\Pi_{\mathrm{ref}}(n_{\mathrm{nC7}}, n_{\mathrm{C6m2}})$: the
probability of finding exactly $n_{\mathrm{nC7}}$ heptane and
$n_{\mathrm{C6m2}}$ 2-methylhexane molecules in the represented framework mass,
at 552 K, with both components at 1 bar. The colour scale is logarithmic so
that the tails are visible; the side panels are the linear marginals.

Two features carry the physics. The cloud sits well away from both the origin
and saturation, and it is **tilted**: high heptane occupancy comes with low
2-methylhexane occupancy, because they compete for the same sites. That
anticorrelation *is* the selectivity, stored as a shape rather than as a mixing
rule. The dashed line marks the strong family's capacity, $n_{\mathrm{nC7}} +
n_{\mathrm{C6m2}} = N_1$; the probability beyond it is the weak family filling
up.

In [ ]:
first_marginal = PI_REFERENCE.sum(axis=1)
second_marginal = PI_REFERENCE.sum(axis=0)
edges = np.append(MESH - builder.DELTA_N / 2, MESH[-1] + builder.DELTA_N / 2)
strong_sites = builder.SITE_FAMILIES[0][0]
colormap = plt.get_cmap("magma").copy()
colormap.set_bad("0.94")

fig = plt.figure(figsize=(8.4, 7.2), constrained_layout=True)
grid = fig.add_gridspec(2, 2, width_ratios=(5, 1.3), height_ratios=(1.3, 5))
main = fig.add_subplot(grid[1, 0])
top = fig.add_subplot(grid[0, 0], sharex=main)
right = fig.add_subplot(grid[1, 1], sharey=main)

image = main.pcolormesh(
    edges, edges, np.ma.masked_where(PI_REFERENCE <= 0.0, PI_REFERENCE).T,
    cmap=colormap, shading="flat", norm=LogNorm(vmin=1e-9, vmax=PI_REFERENCE.max()),
)
main.plot([0, strong_sites], [strong_sites, 0], color="0.35", linestyle="--", linewidth=1.0)
main.plot(*reference_mean, marker="+", color="white", markersize=14, markeredgewidth=2.0, linestyle="none")
main.set_xlabel(r"$n_{\mathrm{nC7}}$  [particles]")
main.set_ylabel(r"$n_{\mathrm{C6m2}}$  [particles]")
main.set_xlim(0, builder.N_MAX)
main.set_ylim(0, builder.N_MAX)
main.set_aspect("equal")
fig.colorbar(image, ax=main, label=r"$\Pi_{\mathrm{ref}}(\mathbf{n})$", location="bottom",
             fraction=0.045, pad=0.02)

top.step(MESH, first_marginal, where="mid", color="tab:blue")
top.fill_between(MESH, first_marginal, step="mid", alpha=0.25, color="tab:blue")
top.set_ylabel("marginal")
top.tick_params(labelbottom=False)
right.step(second_marginal, MESH, where="mid", color="tab:red")
right.fill_betweenx(MESH, second_marginal, step="mid", alpha=0.25, color="tab:red")
right.set_xlabel("marginal")
right.tick_params(labelleft=False)
for axis in (top, right):
    axis.grid(alpha=0.2)

fig.suptitle(
    f"Reference macrostate particle distribution\n"
    f"BEA, {REFERENCE_TEMPERATURE:g} K, $f_i^{{\\mathrm{{ref}}}}$ = {REFERENCE_FUGACITY:g} Pa, "
    f"{int((PI_REFERENCE > 0).sum())} populated macrostates"
)
plt.show()

beyond = PI_REFERENCE[np.add.outer(MESH, MESH) > strong_sites].sum()
print(f"probability beyond n_total = {strong_sites} (the strong family's capacity): {beyond:.4f}")

## 4. Reweighting is the whole method

Ruptura never rebuilds the distribution. It shifts the one it was given:

$$\log \widetilde{\Pi}(\mathbf n;f,\mathbf y,V,T)=\log \Pi_{\mathrm{ref}}(\mathbf n)
+\sum_j n_j\log y_j
+N\log\frac{f}{f^{\mathrm{ref}}}
+N\log\frac{\beta}{\beta^{\mathrm{ref}}}
-\langle H\rangle_{\mathbf n,\mathrm{ref}}\bigl(\beta-\beta^{\mathrm{ref}}\bigr),$$

renormalizes, and takes the mean. Every term is a cheap per-macrostate
multiplication. The last term needs the conditional mean energy, which a Monte
Carlo distribution supplies as a second column; here the model is isothermal and
that term is identically zero — which is also why the temperature dependence of
section 1 lives in the Langmuir parameters rather than in the distribution.

Notice what the composition term does: raising $y_j<1$ to the power $n_j$
penalizes macrostates that are *rich* in component $j$. Competition is not
imposed — it falls out of the exponent.

The panels below reweight the same reference distribution to three total
fugacities at equimolar composition. The cloud slides out of the corner, crosses
the reference state, and piles up against the saturation boundary, becoming
visibly narrower and more anticorrelated as it goes.

In [ ]:
showcase = (1.0e4, 1.0e5, 1.0e6)
equimolar = np.array([0.5, 0.5])

fig, axes = plt.subplots(1, len(showcase), figsize=(13.5, 5.0), sharex=True, sharey=True, constrained_layout=True)
for axis, fugacity in zip(axes, showcase):
    weights = reweight(equimolar, fugacity)
    image = axis.pcolormesh(
        edges, edges, np.ma.masked_where(weights <= 0.0, weights).T,
        cmap=colormap, shading="flat", norm=LogNorm(vmin=1e-9, vmax=weights.max()),
    )
    means = mean_particle_numbers(weights)
    loading = means / PARTICLES_PER_MOL_PER_KG
    axis.plot([0, strong_sites], [strong_sites, 0], color="0.35", linestyle="--", linewidth=0.9)
    axis.plot(*means, marker="+", color="white", markersize=13, markeredgewidth=2.0, linestyle="none")
    axis.set_title(f"$f$ = {fugacity:g} Pa, $y_i$ = 0.5\n$q$ = ({loading[0]:.3f}, {loading[1]:.3f}) mol/kg")
    axis.set_xlabel(r"$n_{\mathrm{nC7}}$")
    axis.set_xlim(0, builder.N_MAX)
    axis.set_ylim(0, builder.N_MAX)
    axis.set_aspect("equal")
axes[0].set_ylabel(r"$n_{\mathrm{C6m2}}$")
fig.colorbar(image, ax=axes, label=r"$\widetilde{\Pi}(\mathbf{n})$", fraction=0.03)
plt.show()

## 5. Isotherms: MPD against dual-site Langmuir + IAST

Now hand the distribution to Ruptura. Two `MixturePrediction` runs cover 64
points logarithmically spaced from $10^3$ to $10^7$ Pa:

| run | directory | model |
| --- | --- | --- |
| MPD | `mixture/mpd` | `MixturePredictionMethod: "MPD"`, reading `particle_distribution.data` |
| reference | `mixture/iast` | the same fitted dual-site Langmuir isotherms, mixed by IAST |

Both runs report a pure-component and a mixture curve in the same file. For the
MPD run the pure-component column is the distribution reweighted with a *one-hot*
composition — $\mathbf y = [1,0]$ and $\mathbf y = [0,1]$ — which is exactly the
statement that a macrostate containing any molecule of the absent species has
zero probability. The mixture column uses $\mathbf y = [0.5, 0.5]$.

In [ ]:
MIXTURE_RUNS = {"MPD": EXAMPLE_DIR / "mixture" / "mpd", "Langmuir/IAST": EXAMPLE_DIR / "mixture" / "iast"}

mixture_timings = {label: run_ruptura(directory) for label, directory in MIXTURE_RUNS.items()}
mixture_data = {
    label: {component: read_mixture_prediction(directory, index, component)
            for index, component in enumerate(COMPONENTS)}
    for label, directory in MIXTURE_RUNS.items()
}

for label, record in mixture_timings.items():
    print(f"{label:>13}: {record['wall_time_s']:8.3f} s wall")

In [ ]:
COMPONENT_COLORS = {"nC7": "tab:blue", "C6m2": "tab:red"}
STYLES = {
    "MPD": dict(linestyle="-", marker="o", filled=True),
    "Langmuir/IAST": dict(linestyle="--", marker="s", filled=False),
}


def draw(axis, x, y, label, component, *, markevery):
    style = STYLES[label]
    color = COMPONENT_COLORS[component]
    axis.plot(
        x, y, color=color, linestyle=style["linestyle"], marker=style["marker"],
        markerfacecolor=color if style["filled"] else "none", markeredgecolor=color,
        markersize=5, markevery=markevery, linewidth=1.6, label=f"{component}, {label}",
    )


fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0), sharey=True, constrained_layout=True)
for axis, key, title in zip(axes, ("pure", "mixture"), ("Pure components", "Equimolar mixture, $y_i$ = 0.5")):
    for label in STYLES:
        for component in COMPONENTS:
            curve = mixture_data[label][component]
            draw(axis, curve["pressure"], curve[key], label, component, markevery=3)
    axis.set_xscale("log")
    axis.set_xlabel("fugacity  [Pa]" if key == "pure" else "total fugacity  [Pa]")
    axis.set_title(title)
    axis.grid(alpha=0.2)
axes[0].set_ylabel("loading  [mol/kg]")
axes[1].legend(loc="upper left", fontsize=9, framealpha=0.9)
fig.suptitle(f"BEA at {REFERENCE_TEMPERATURE:g} K — MPD (solid, filled) vs dual-site Langmuir/IAST (dashed, open)")
plt.show()

The pure-component curves lie on top of each other to plotting accuracy: with a
one-hot composition there is no mixing rule to disagree about, so the reweighted
multinomial reproduces the fitted dual-site Langmuir isotherm exactly. This is
the sanity check that the translation was faithful.

The mixture curves separate. MPD is the *exact* competitive result for this site
model — mean occupancies of independent sites — whereas IAST solves a
thermodynamic consistency condition on the pure-component spreading pressures.
The two agree in the Henry regime, where both reduce to $q_i=K_if_i$, and part
company as the strong family fills, because the two families have opposite
selectivities — the strong one prefers nC7, the weak one prefers C6m2 — and IAST
has to reconcile them through a single spreading pressure.

## 6. Does the mesh cost anything?

The distribution is tabulated with $\Delta n=2$, so Ruptura sees 5151
macrostates instead of the 20301 that $\Delta n=1$ would give — a fourfold
saving in every equilibrium evaluation, which matters a great deal once the
column solver starts calling it millions of times.

The price is that the mesh has to resolve the reweighted distribution.
Comparing the MPD loadings against the analytical competitive dual-site Langmuir
expression — which the multinomial reproduces *exactly* on a unit mesh —
isolates that error.

In [ ]:
probe = np.logspace(2, 8, 25)
rows = []
for fugacity in probe:
    exact = competitive_langmuir(np.array([0.5, 0.5]) * fugacity)
    predicted = mpd_loading(np.array([0.5, 0.5]), fugacity)
    rows.append((fugacity, *(np.abs(predicted - exact) / np.maximum(exact, 1e-30))))

print(f"{'f [Pa]':>10} {'rel. err nC7':>14} {'rel. err C6m2':>15}")
for fugacity, first_error, second_error in rows:
    marker = "   <- plotted range" if 1.0e3 <= fugacity <= 1.0e7 else ""
    print(f"{fugacity:10.2e} {first_error:14.2e} {second_error:15.2e}{marker}")

inside = [row for row in rows if 1.0e3 <= row[0] <= 1.0e7]
print(f"\nMax relative discretization error over the plotted range: "
      f"{max(max(row[1], row[2]) for row in inside):.2e}")

Inside $10^3$–$10^7$ Pa the mesh is essentially free: the error drops to machine
precision within half a decade, because the reweighted distribution is many mesh
spacings wide. It only reappears where the distribution collapses onto one or
two macrostates — the deep Henry limit, where the population is a fraction of a
molecule, and full saturation, where every site is occupied. Both are avoidable
by choosing the tabulated window sensibly, and both would disappear at
$\Delta n=1$.

## 7. Breakthrough

The same distribution now drives a packed-bed simulation. The column, feed and
integrator settings are those of `examples/BEA-alkanes-C7/breakthrough` — 552 K,
1.2 bar inlet, 0.019 m/s, 0.1 m of BEA, CVODE, fixed-pressure inlet velocity,
`NumberOfTimeSteps: "auto"` — with 1 % nC7 and 1 % C6m2 in helium instead of
seven isomers.

Two settings differ from that example, and only for cost:
`NumberOfGridPoints` is 20 rather than 100, and the output `TimeStep` is 0.105 s
rather than 0.001 s. These are the values used by the existing MPD benchmark in
`examples/MPD/BEA-alkanes/breakthrough`, and the reason is section 8: an MPD
equilibrium evaluation walks every populated macrostate, so its cost scales with
(grid points) × (integration steps). Both models here use the identical column
and identical settings, so the comparison is unaffected.

> **This cell is the long one.** The Langmuir/IAST run finishes in seconds; the
> MPD run takes on the order of an hour on a laptop. Results are cached in each
> directory's `benchmark.json`, so re-running the notebook is instant. Pass
> `force=True` to recompute.

In [ ]:
BREAKTHROUGH_RUNS = {
    "MPD": EXAMPLE_DIR / "breakthrough" / "mpd",
    "Langmuir/IAST": EXAMPLE_DIR / "breakthrough" / "iast",
}

breakthrough_timings = {}
for label, directory in BREAKTHROUGH_RUNS.items():
    print(f"Running {label} in {directory.relative_to(EXAMPLE_DIR)} ...", flush=True)
    breakthrough_timings[label] = run_ruptura(directory)
    print(f"  -> {breakthrough_timings[label]['total_time_s']:.3f} s", flush=True)

breakthrough_data = {
    label: {component: read_outlet(directory, index, component)
            for index, component in enumerate(COMPONENTS, start=1)}
    for label, directory in BREAKTHROUGH_RUNS.items()
}

In [ ]:
fig, axis = plt.subplots(figsize=(9.5, 5.6), constrained_layout=True)
for label in STYLES:
    for component in COMPONENTS:
        curve = breakthrough_data[label][component]
        draw(axis, curve["minutes"], curve["normalized"], label, component, markevery=20)
axis.axhline(1.0, color="0.7", linewidth=0.8, zorder=0)
axis.set_xlabel("time  [min]")
axis.set_ylabel(r"$p_i / (p_t\, y_{i,0})$  [-]")
axis.set_xlim(left=0.0)
axis.set_ylim(-0.02, 1.2)
axis.grid(alpha=0.2)
axis.legend(loc="lower right", fontsize=9)
axis.set_title("BEA breakthrough, 1 % nC7 / 1 % C6m2 in helium — MPD (solid, filled) vs Langmuir/IAST (dashed, open)")
plt.show()

Both models put 2-methylhexane through first and hold n-heptane back, and the
roll-up overshoot of C6m2 above unity — displaced from the sites as the heptane
front arrives — is reproduced by both. The offset between the two sets of curves
traces directly back to the mixture isotherms of section 5: wherever the
multinomial holds more of a component than IAST predicts, that is extra capacity
the front has to fill before it can leave the column.

That the two nearly coincide is the point worth keeping. The MPD run was handed
no isotherm at all — just a table of probabilities — and the column solver
recovered the same separation.

## 8. What it costs

Ruptura reports its own `Total time` only for breakthrough runs, so the
mixture-prediction rows fall back to wall time.

In [ ]:
def equilibrium_model(directory: Path) -> str:
    settings = json.loads((Path(directory) / "simulation.json").read_text(encoding="utf-8"))
    if "MPDSettings" in settings:
        return f"{int((PI_REFERENCE > 0).sum())} macrostates"
    sites = sum(
        len(component.get("PhysisorptionSites", []))
        for component in settings["Components"]
        if not component.get("CarrierGas", False)
    )
    return f"{sites} Langmuir sites"


header = f"{'simulation':<16} {'model':<14} {'equilibrium model':<18} {'ruptura [s]':>12} {'wall [s]':>10}"
print(header)
print("-" * len(header))
for kind, runs, timings in (
    ("mixture", MIXTURE_RUNS, mixture_timings),
    ("breakthrough", BREAKTHROUGH_RUNS, breakthrough_timings),
):
    for label, directory in runs.items():
        record = timings[label]
        reported = f"{record['total_time_s']:.3f}" if np.isfinite(record["total_time_s"]) else "n/a"
        print(f"{kind:<16} {label:<14} {equilibrium_model(directory):<18} "
              f"{reported:>12} {record['wall_time_s']:>10.3f}")

for kind, timings in (("mixture", mixture_timings), ("breakthrough", breakthrough_timings)):
    reference = timings["Langmuir/IAST"]["wall_time_s"]
    print(f"\n{kind}: MPD is {timings['MPD']['wall_time_s'] / reference:.1f}x the Langmuir/IAST wall time")

sub_timings = breakthrough_timings["MPD"].get("sub_timings_s", {})
if sub_timings:
    print("\nMPD breakthrough, Ruptura's internal split:")
    for name, value in sub_timings.items():
        print(f"  {name:<28} {value:10.3f} s")

The mixture prediction costs essentially nothing either way — 64 equilibrium
evaluations. The breakthrough is where the difference lands, and the reason is
structural rather than incidental: a competitive Langmuir or IAST evaluation
touches a handful of parameters, while an MPD evaluation sums over every
populated macrostate, at every grid point, at every internal integration step.
The cost is linear in the number of macrostates, which is why the tabulated
window and $\Delta n$ are the knobs that matter — and why $\Delta n=2$ was worth
the check in section 6.

That is the trade. What you buy is an equilibrium model with no mixing rule and
no fitted mixture isotherm: for a real system the distribution comes from Monte
Carlo and carries the actual adsorbate–adsorbate correlations, including the ones
no analytical isotherm has a term for. Here we could write it down by hand,
which is precisely what let us check that the machinery is exact.

## 9. Summary

In [ ]:
print("Pure components — max |MPD - Langmuir| over 64 points:")
for component in COMPONENTS:
    mpd_curve = mixture_data["MPD"][component]["pure"]
    reference_curve = mixture_data["Langmuir/IAST"][component]["pure"]
    scale = np.maximum(np.abs(reference_curve), 1e-12)
    print(f"  {component:>5}: {np.abs(mpd_curve - reference_curve).max():.3e} mol/kg "
          f"({np.max(np.abs(mpd_curve - reference_curve) / scale):.2e} relative)")

print("\nEquimolar mixture — max |MPD - IAST| over 64 points:")
for component in COMPONENTS:
    mpd_curve = mixture_data["MPD"][component]["mixture"]
    reference_curve = mixture_data["Langmuir/IAST"][component]["mixture"]
    scale = np.maximum(np.abs(reference_curve), 1e-12)
    worst = int(np.argmax(np.abs(mpd_curve - reference_curve) / scale))
    print(f"  {component:>5}: {np.abs(mpd_curve - reference_curve).max():.3e} mol/kg "
          f"({np.max(np.abs(mpd_curve - reference_curve) / scale):.2%} relative, "
          f"worst at {mixture_data['MPD'][component]['pressure'][worst]:.3g} Pa)")

print("\nBreakthrough — first crossing of 50 % of the feed [min]:")
for component in COMPONENTS:
    times = {label: first_crossing(breakthrough_data[label][component]) for label in BREAKTHROUGH_RUNS}
    print(f"  {component:>5}: MPD {times['MPD']:7.2f}, Langmuir/IAST {times['Langmuir/IAST']:7.2f}, "
          f"difference {times['MPD'] - times['Langmuir/IAST']:+.2f}")

**What this example established**

1. GCMC isotherms can be fitted directly in the form a competitive site model
   needs — one saturation capacity per site family, shared by the components —
   at a cost of about 1 % of the largest loading against the unconstrained fit.
   $\Delta H$ is not fittable from single-temperature data and comes from the
   simulated heat of desorption instead.
2. A multisite Langmuir isotherm with a shared saturation capacity per family is
   the mean of a product of multinomial distributions. Writing that distribution
   down converts an isotherm into an MPD with no loss.
3. The framework mass is the dictionary between particle counts and loadings.
   Fixing the window at 200 particles and splitting it in proportion to the
   fitted capacities puts both components on a $[0,200]$ axis with the reference
   state in the interior.
4. Reweighting recovers the analytical isotherm to machine precision wherever
   the tabulated mesh resolves the distribution, so the pure-component curves
   coincide and every remaining difference against IAST is the mixing rule, not
   the method.
5. The column solver treats an MPD exactly like any other equilibrium model.
   The breakthrough curves agree closely; the residual offset traces back to the
   isotherm difference.
6. MPD costs what it costs: one pass over the populated macrostates per
   equilibrium evaluation. Keep the window tight and $\Delta n$ as coarse as the
   distribution allows.

**Where to go next.** Replace `particle_distribution.data` with a distribution
from NVT+$W$ or TMMC on a real framework, add the conditional mean Hamiltonian
$\langle H\rangle_{\mathbf n,\mathrm{ref}}$ as a second column, and the same
input files run non-isothermally — the temperature term of the reweighting stops
being zero and the column solver needs no other change.